In [1]:
# You can copy the folder structure of the previous analysis in this way:

# rsync -av -f"+ */" -f"- *" "/nfs/lab/projects/islet_multiomics_stress/analysis/CellChat/Results/May2022/" "/nfs/lab/projects/islet_multiomics_stress/analysis/CellChat/Results/May2022.SCT"

# Installation
It's suggested to do directly on terminal using the same R version of the notebook

In [2]:
#devtools::install_github("sqjin/C§ellChat")
#install.packages('NMF')
#devtools::install_github("jokergoo/circlize")
#devtools::install_github("jokergoo/ComplexHeatmap")

## Load reticulate library and import anndata module
The anndata module in python is used to load .h5ad files in R </br>
Install the anndata module in python withing the correct conda enviroment before launching.

In [3]:
#essential reticulate functions (must run first)
Sys.setenv(RETICULATE_PYTHON="/home/luca/anaconda3/envs/reticulate/bin/python")
library(reticulate)
reticulate::use_python("/home/luca/anaconda3/envs/reticulate/bin/python")
reticulate::use_condaenv("/home/luca/anaconda3/envs/reticulate")
reticulate::py_module_available(module='anndata') #needs to be TRUE
reticulate::import('anndata') #good to make sure this doesn't error

[1] TRUE

Module(anndata)

# Setup

## Load libraries

In [4]:
pacman::p_load(CellChat, Seurat, patchwork, NMF, ggalluvial, dplyr, logr, parallel)

## Working Directories and Options

In [5]:
# Input files
Combine.Seurat = "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/Assets/20230518_RNA_FiltMin20exceptBetaLate_CellChat_nPODids.SCT.rds.SM.Lpct10.rds"
# Variables
condition.ls = c("ND", "Aab", "T1D_early","T1D_late")

# Output dirs
out.dir <- "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/Results/corrected.permutation.test.100/"

options(stringsAsFactors = FALSE)
# start log
options("logr.on" = TRUE, "logr.notes" = TRUE)
options("logr.autolog" = TRUE)
options("logr.compact" = TRUE)
options("logr.traceback" = TRUE)
log.file = paste(out.dir, Sys.Date(),".uncorrected.Permutation.test.log", sep="")

In [6]:
log_open(log.file)

[1] "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/Results/corrected.permutation.test.100/log/2023-08-22.uncorrected.Permutation.test.log"

# Pre Processing

In [7]:
log_print("Loading Seurat obj")
seurat_object <- readRDS(Combine.Seurat)

[1] "Loading Seurat obj"


In [8]:
# Check cell pops id
log_print("Changing idents")
seurat_object <- SetIdent(seurat_object, value = seurat_object$celltype)

[1] "Changing idents"


In [9]:
group.sizes = data.frame(table(seurat_object$samples, seurat_object$condition))
colnames(group.sizes) = c("sample", "Condition", "Freq")
ND.size = nrow(group.sizes[group.sizes$Condition == condition.ls[1] & group.sizes$Freq > 0,])
AAB.size = nrow(group.sizes[group.sizes$Condition == condition.ls[2] & group.sizes$Freq > 0,])
T1D.size = nrow(group.sizes[group.sizes$Condition == condition.ls[3] & group.sizes$Freq > 0,])
T2D.size = nrow(group.sizes[group.sizes$Condition == condition.ls[4] & group.sizes$Freq > 0,])
ND.size + AAB.size + T1D.size + T2D.size

[1] 31

In [10]:
# Settings for CellChat
## Set boot number for cellchat
nboot.cellchat = 100
## Set workers (Processors for cellchat)
workers = 15
## Increase RAM cap for celchat
options(future.globals.maxSize= 891289600)
## Set trim for cellchat
trim = 0
# Settings for pemutation test
## Set number of permutations
nboot = 100
## Set group sizes
group.sizes = c(ND.size,AAB.size,T1D.size,T2D.size)

# Set cores for multi.cellchat
cores = 4 # I use 4 because I have 4 conditions

In [11]:
# NOTE: USING MIN CELLS = 1

In [12]:
setwd(out.dir)

In [13]:
# U can change the rna counts matrix down-here

In [14]:
multi.CellChat = function(mcla.i = 1){
    gc(reset = TRUE)
    # Setup variables
    seurat_object.use = seurat_object.ls[[mcla.i]]
    condition.use = condition.ls[[mcla.i]]
    log_print(paste("Starting CellChat analysis on: ", condition.use))
    # extract RNA data
    data.input <- GetAssayData(seurat_object.use, assay = "RNA", slot = "data") #get normalized data
    # Create CellChat Obj
    log_print(paste("Creating CellChat Obj: ", condition.use))
    labels <- Idents(seurat_object.use)
    meta <- data.frame(group = labels, row.names = names(labels)) # create a dataframe of the cell labels
    cellchat <- createCellChat(object = data.input, group.by = Idents(seurat_object.use))
    cellchat <- addMeta(cellchat, meta = meta, meta.name = "labels")
    cellchat <- setIdent(cellchat, ident.use = "labels")
    groupSize <- as.numeric(table(cellchat@idents)) # number of cells in each cell group
    CellChatDB <- CellChatDB.human # use CellChatDB.mouse if running on mouse data
    CellChatDB.use <- CellChatDB.human # use whole database
    cellchat@DB <- CellChatDB.use
    gc(reset = TRUE)
    # Running CellChat analysis
    log_print(paste("Identify interactions: ", condition.use))
    future::plan("multicore", workers = workers)
    cellchat <- subsetData(cellchat) # This step is necessary even if using the whole database
    cellchat <- identifyOverExpressedGenes(cellchat)
    cellchat <- identifyOverExpressedInteractions(cellchat)
    #cellchat <- projectData(cellchat, PPI.human)
    gc(reset = TRUE)
    log_print(paste("Compute IS: ", condition.use))
    future::plan("multicore", workers = workers)
    options(future.globals.maxSize= 891289600)
    cellchat <- computeCommunProb(cellchat, type = "truncatedMean", trim = trim, nboot = nboot.cellchat)
    gc(reset = TRUE)
    log_print(paste("Filter results: ", condition.use))
    # Filter out the cell-cell communication if there are only few number of cells in certain cell groups
    cellchat <- filterCommunication(cellchat, min.cells = 1)
    cellchat <- computeCommunProbPathway(cellchat)
    cellchat <- aggregateNet(cellchat)
    gc(reset = TRUE)
    log_print(paste("Save dfnet: ", condition.use))
    df.net <- subsetCommunication(cellchat,  thresh = 1)
    df.net.file <-  paste(condition.use, "_df.net_nboot", i, "_trim", trim, ".txt", sep="")
    return(write.table(df.net, file=df.net.file, sep = "\t", quote = F, col.names=T, row.names = F))
    log_print(paste("Saved: ", condition.use))
}

In [15]:
# NBOOT FROM 24 to 100

In [ ]:
log_print(" - Starting analysis ")
## Create samples variable for log
samples = levels(factor(seurat_object$samples))
log_print(paste(" - Samples detected: ", length(samples)))
log_print(paste(" - Group sizes: ", "\n",
          "\t CTR:", group.sizes[1], "\n",
          "\t AAB:", group.sizes[2], "\n",         
          "\t ETD:", group.sizes[3], "\n",          
          "\t LTD:", group.sizes[4], "\n"))

cl <- makeCluster(cores)
setDefaultCluster(cl)

for (i in 24:nboot) {
    set.seed(i)
    # Message run number
    log_print(paste(" - Permutation number ", i))
    log_print(paste(" - Split seurat obj "))
    # Get sample list
    samples = levels(factor(seurat_object$samples))
    # Reset list of seurat objects
    seurat_object.ls = vector(mode = "list", length = length(condition.ls))
    for (condition.i in seq_along(condition.ls)){
        gc(reset = TRUE)
        log_print(paste(" - Processing ", condition.ls[condition.i]))
        samples.use = sample(samples, size = group.sizes[condition.i], replace = F)
        # Remove those samples from original list
        samples = samples[! samples %in% samples.use]
        log_print(paste(" - Selected ", length(samples.use), " random samples, ", length(samples), " remaining"))
        seurat_object.ls[[condition.i]] = subset(x = seurat_object, subset = samples %in%  samples.use)
        log_print(paste(" - Sample used: ", samples.use))
    }  
    mclapply(1:length(condition.ls), function(mcla.i) multi.CellChat(mcla.i), mc.cores = cores)
}
             
# Clean up the parallel backend
setDefaultCluster(NULL)
stopCluster(cl)

[1] " - Starting analysis "
[1] " - Samples detected:  31"
[1] " - Group sizes:  \n \t CTR: 10 \n \t AAB: 9 \n \t ETD: 7 \n \t LTD: 5 \n"
